# Συνοπτικά: Φιλτράρισμα Μετά από Ομαδοποίηση (Filtering After Grouping)

Το **Filtering After Grouping** είναι η τεχνική όπου **πρώτα ομαδοποιούμε** τα δεδομένα για να υπολογίσουμε ένα στατιστικό μέγεθος (άθροισμα, μέσο όρο, πλήθος) και **μετά φιλτράρουμε** το αποτέλεσμα με βάση αυτό το μέγεθος.

> 💡 Στην **SQL**, η τεχνική αυτή αντιστοιχεί ακριβώς στο **`HAVING` clause**.

---

### 🔍 Φιλτράρισμα ΠΡΙΝ vs ΜΕΤΑ το Group By

*   **ΠΡΙΝ (`WHERE` στην SQL):** Φιλτράρει μεμονωμένες γραμμές από τον αρχικό πίνακα.
    *   *Παράδειγμα:* "Κράτα μόνο τις πωλήσεις που έγιναν στην **Αθήνα**."
*   **ΜΕΤΑ (`HAVING` στην SQL):** Φιλτράρει ολόκληρες ομάδες με βάση το τελικό τους σκορ.
    *   *Παράδειγμα:* "Κράτα μόνο τα καταστήματα που έκαναν **συνολικές πωλήσεις > 10.000€**."

---

### ⚙️ Η Ροή των 3 Βημάτων στα Pandas

1.  **`.groupby().ανασύνθεση()`**: Ομαδοποίηση και υπολογισμός (π.χ. `.sum()`, `.count()`).
2.  **`.reset_index(name="...")`**: Μετατροπή σε κανονικό DataFrame και **ονοματοδοσία** της νέας στήλης.
3.  **Φιλτράρισμα**: Εφαρμογή του φίλτρου πάνω στη νέα στήλη (με αγκύλες ή με `.query()`).

---

### 💻 Παράδειγμα Κώδικα (Ολοκληρωμένο)

```python
# Βήμα 1 & 2: Ομαδοποίηση και ονοματοδοσία της νέας στήλης 'total_sales'
store_sales = df.groupby('store')['sales_amount'].sum().reset_index(name='total_sales')

# Βήμα 3: Φιλτράρισμα ΜΕΤΑ το groupby
top_stores = store_sales[store_sales['total_sales'] > 10000]
```

### 💎 Η "Pro" Εναλλακτική σε 1 Γραμμή (Με `.query()`)
```python
df.groupby('store')['sales_amount'].sum().reset_index(name='total_sales').query('total_sales > 10000')
```


In [2]:
import pandas as pd

In [3]:
movies= pd.read_csv('../data/movies.csv', index_col=0)

In [4]:
movies.info()

<class 'pandas.DataFrame'>
Index: 500 entries, Broken Verdict to Last Signal IIII
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   year          500 non-null    int64  
 1   genre         500 non-null    str    
 2   runtime_min   500 non-null    int64  
 3   budget_musd   500 non-null    float64
 4   revenue_musd  500 non-null    float64
 5   rating        500 non-null    float64
 6   votes         500 non-null    int64  
dtypes: float64(3), int64(3), str(1)
memory usage: 31.2+ KB


In [5]:
genre_count=(
    movies.groupby('genre')['genre']
    .count()
    .reset_index(name='movies_count')
)
genre_count[genre_count['movies_count']<=80]

# SELECT COUNT(genre)
# FROM movies
# GROUP BY genre
# HAVING COUNT(genre)<=80

,genre,movies_count
0,Action,79
1,Animation,70


In [6]:
# ο τρόπος του .query μονο για το βήμα φιλτραρίσματος γραμμών ή φιλτράρισμα groups
(
    movies.groupby('genre')['genre']
    .count()
    .reset_index(name='movies_count')
    .query("movies_count<=80") #μόνο για το φιλτράρισμα δηλαδή where & having αλλά χρησιμοποιώντας python σύνταξη
)

,genre,movies_count
0,Action,79
1,Animation,70


In [7]:
# στην ουσία το genre_count είναι το cte μου και το result το main query
genre_count= movies.groupby('genre')['genre'].count().reset_index(name='movies_count')

result= genre_count[genre_count['movies_count'].between(60,80)]
result

,genre,movies_count
0,Action,79
1,Animation,70


In [8]:
(
    movies.groupby('genre')['genre']
    .count()
    .reset_index(name='movies_per_genre')
    .query("movies_per_genre.between(60,80)")
)

,genre,movies_per_genre
0,Action,79
1,Animation,70


In [9]:
movies.query("genre=='Action' & runtime_min<90")[['runtime_min']]

# movies[(movies['genre']=='Action')&(movies['runtime_min']<90)][['runtime_min']]


,runtime_min
title,
Electric Machine,86
Winter Compass,64
Crimson Harvest,64
Last Archive,85
Hollow Verdict II,86
Paper Circuit II,64
Quiet Orchard III,66
Northern Circuit III,64
Silent Compass III,78


In [10]:
#ποτέ δεν αναμειγνύω .loc(γραμμές, στήλες) με το .query (επιστρέφει έτοιμο DataFrame)
(
    movies.query("genre=='Action' and year.between(2015,2025)")
    [['genre','runtime_min','year','revenue_musd']]
    .reset_index()
)

,title,genre,runtime_min,year,revenue_musd
0,Golden Signal,Action,144,2025,97.2
1,Electric Machine,Action,86,2022,45.4
2,Winter Compass,Action,64,2018,268.0
3,Crimson Verdict,Action,106,2016,74.8
4,Crimson Harvest,Action,64,2017,12.8
5,Glass Archive,Action,101,2018,376.8
6,Last Machine,Action,123,2022,19.8
7,Broken Orchard,Action,126,2023,0.5
8,Hollow Verdict II,Action,86,2021,279.8
9,Quiet River II,Action,97,2017,163.1


In [11]:
movies.loc[
    (movies['genre']=='Action') &
    (movies['year'].between(2015,2025)),
    ['genre','runtime_min','year','revenue_musd']
].reset_index()

,title,genre,runtime_min,year,revenue_musd
0,Golden Signal,Action,144,2025,97.2
1,Electric Machine,Action,86,2022,45.4
2,Winter Compass,Action,64,2018,268.0
3,Crimson Verdict,Action,106,2016,74.8
4,Crimson Harvest,Action,64,2017,12.8
5,Glass Archive,Action,101,2018,376.8
6,Last Machine,Action,123,2022,19.8
7,Broken Orchard,Action,126,2023,0.5
8,Hollow Verdict II,Action,86,2021,279.8
9,Quiet River II,Action,97,2017,163.1


### επαναληπτικές ασκήσεις στον πίνακα students

In [12]:
students = pd.read_csv("../data/students.csv", index_col=0)

In [30]:
students.head(3)

,study_hours_week,attendance_pct,prior_score,tutoring,sleep_hours,final_score,result
student_id,,,,,,,
S3000,4.9,71,92,1,6.6,48,pass
S3001,11.8,75,72,0,7.6,48,pass
S3002,11.3,100,84,1,5.7,62,pass


In [35]:
# βρες τους μαθητές που έχουν κοπεί (όλες τις στήλες)

#students[students['result']=='fail']
students.query(" result=='fail' ")

,study_hours_week,attendance_pct,prior_score,tutoring,sleep_hours,final_score,result
student_id,,,,,,,
S3003,8.3,92,42,1,7.5,23,fail
S3008,8.2,73,61,1,7.4,30,fail
S3010,11.3,100,48,0,6.2,32,fail
S3014,10.6,73,63,0,7.4,31,fail
S3015,12.5,95,53,0,5.6,34,fail
...,...,...,...,...,...,...,...
S3787,8.6,76,77,0,6.8,39,fail
S3793,2.4,98,79,0,6.5,34,fail
S3794,6.8,86,56,0,7.3,39,fail


In [37]:
# Βρείτε τους μαθητές που διάβαζαν πάνω από 12 ώρες την εβδομάδα 
# και είχαν ποσοστό παρουσιών τουλάχιστον 90%.

#students[(students['study_hours_week']>12)&(students['attendance_pct']>90)]
students.query(" study_hours_week>12 and attendance_pct>90 ")

,study_hours_week,attendance_pct,prior_score,tutoring,sleep_hours,final_score,result
student_id,,,,,,,
S3007,15.9,92,66,0,3.5,66,pass
S3015,12.5,95,53,0,5.6,34,fail
S3041,13.5,95,55,0,8.1,45,pass
S3059,13.0,97,62,1,7.8,63,pass
S3067,14.6,97,63,0,7.8,58,pass
S3080,14.5,91,81,1,7.6,73,pass
S3084,15.6,92,70,0,6.5,59,pass
S3086,15.4,91,67,1,6.6,69,pass
S3087,12.7,100,57,1,8.0,69,pass


In [ ]:
#Εμφανίστε μόνο τις στήλες prior_score και final_score για όσους μαθητές 
# έκαναν ιδιαίτερα μαθήματα (tutoring == 1).

students.loc[students['tutoring']==1, ['prior_score','final_score']]
#students.query(" tutoring==1 ")[['prior_score','final_score']]

,prior_score,final_score
student_id,,
S3000,92,48
S3002,84,62
S3003,42,23
S3004,83,60
S3008,61,30
...,...,...
S3778,72,46
S3781,67,40
S3789,75,61


In [51]:
#Υπολογίστε τον μέσο όρο της τελικής βαθμολογίας (final_score) ανάλογα με το αν οι μαθητές 
#έκαναν ιδιαίτερα μαθήματα (tutoring) ή όχι, ονομάζοντας τη νέα στήλη average_final_score.

(
    students.groupby('tutoring')['final_score']
    .mean().sort_values(ascending=False)
    .reset_index(name='average_final_score')
)


,tutoring,average_final_score
0,1,47.763547
1,0,42.768844


In [55]:
# Βρείτε πόσες ώρες ύπνου κατά μέσο όρο (sleep_hours) και ποιο ήταν το 
# μέσο τελικό σκορ (final_score) των μαθητών για κάθε ξεχωριστό αποτέλεσμα (result).
# Ονομάστε τις νέες στήλες ανάλογα.

(
    students.groupby('result')
    .aggregate(avg_sleep_hours=('sleep_hours','mean'),
               avg_final_score=('final_score','mean'))
    .round(2)               
    .reset_index()
    .sort_values('avg_final_score', ascending=False)               
)

,result,avg_sleep_hours,avg_final_score
1,pass,6.92,52.53
0,fail,6.75,29.96


In [88]:
# Ομαδοποιήστε τα δεδομένα με βάση τις ώρες ύπνου (sleep_hours). 
# Βρείτε πόσοι μαθητές αντιστοιχούν σε κάθε διάρκεια ύπνου (count) και 
# κρατήστε μόνο τις διάρκειες ύπνου εκείνες που αφορούν τουλάχιστον 15 μαθητές συνολικά.

(
    students.groupby('sleep_hours')['sleep_hours']
    .count()
    .reset_index(name='student_count')
    .query(" student_count>=15 ")
    .sort_values('sleep_hours',ascending=False)
)

,sleep_hours,student_count
46,8.3,15
44,8.1,22
42,7.9,24
41,7.8,24
39,7.6,27
38,7.5,24
37,7.4,27
36,7.3,25
35,7.2,27
34,7.1,26


In [ ]:
# Υπολογίστε το μέσο και το διάμεσο (median) final_score ανάλογα με το αν οι μαθητές 
# έκαναν ιδιαίτερα (tutoring). Κρατήστε στο τελικό αποτέλεσμα μόνο τις ομάδες εκείνες 
# όπου το διάμεσο σκορ είναι αυστηρά μεγαλύτερο από 50, εμφανίζοντας στο τέλος τις στήλες 
# της ομάδας (tutoring) και του διάμεσου σκορ.

(
    students.groupby('tutoring')['final_score']
    .aggregate(avg_final_score='mean',
                median_final_score='median')
    .reset_index()
    .query(" median_final_score>50 ")
    .sort_values('median_final_score', ascending=False)
    [['tutoring','median_final_score']]
)

,tutoring,median_final_score
